In [97]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [98]:
import logging
from random import random, randrange, randint

import torch

In [99]:
logger = logging.Logger(__name__)
logger.setLevel(logging.INFO)

In [100]:
from model import SnakeSolver

In [101]:
def get_device() -> torch.device:
    device_name: str = "cuda" if torch.cuda.is_available() else "cpu"

    logger.info(f"Device name: {device_name}")

    return torch.device(device_name)

In [102]:
from model import select_model_action


def select_action(model: SnakeSolver, state: torch.Tensor, action_size: int, epsilon: float) -> int:
    if random() < epsilon:
        return randrange(action_size)
    
    return select_model_action(model, state, dim=1)

In [103]:
from SnakeGame.Board import Board

def get_random_board() -> Board:
    width: int = randint(6, 25)
    height: int = randint(6, 25)

    if not (width * height) % 2:
        width += 1

    return Board(width, height)

In [104]:
from collections import deque
from dataclasses import dataclass, field
from random import sample


@dataclass
class StateSaver:
    state: torch.Tensor
    action: int
    reward: float
    next_state: torch.Tensor
    done: bool


class Buffer:
    def __init__(self, capacity: int) -> None:
        self.data = deque(maxlen=capacity)
    
    def append(self, value: StateSaver) -> None:
        self.data.append(value)

    def sample(self, batch_size: int) -> list[StateSaver]:
        return sample(self.data, batch_size)

    def __len__(self) -> int:
        return len(self.data)

In [105]:
@dataclass
class EpsilonData:
    epsilon_start: float = 1.0
    current: float = 1.0
    epsilon_end: float = 5e-2
    epsilon_decay: float = 0.995


@dataclass
class TrainData:
    device: torch.device
    model: SnakeSolver
    target_model: SnakeSolver
    optimizer: torch.optim.Optimizer
    transition_buffer: Buffer
    
    gamma: float = 0.95
    batch_size: int = 32
    epsilon_data: EpsilonData = field(default_factory=lambda : EpsilonData())

In [106]:
from SnakeGame.Board import BoardStatus


def get_reward(board: Board, length: int) -> tuple[float, int]:
    result: float = -1.0
    
    if ((new_length := len(board.get_snake())) > length):
        result += 250.0
    
    if board._status == BoardStatus.WIN:
        result += 1500.0
    elif board._status == BoardStatus.LOSE:
        result -= 500.0
    
    return result, new_length

In [107]:
def optimize_model(data: TrainData) -> None:
    if len(data.transition_buffer) < data.batch_size:
        return
    
    transitions = data.transition_buffer.sample(data.batch_size)

    states = torch.stack([transition.state for transition in transitions]).to(data.device)
    actions = torch.tensor(
        [transition.action for transition in transitions],
        dtype=torch.long,
    ).unsqueeze(1).to(data.device)
    rewards = torch.tensor(
        [transition.reward for transition in transitions],
        dtype=torch.float32,
    ).to(data.device)
    next_states = torch.stack([transition.next_state for transition in transitions]).to(data.device)
    dones = torch.tensor(
        [transition.done for transition in transitions],
        dtype=torch.float32,
    ).to(data.device)

    current_q_values = data.model(states).gather(1, actions).squeeze(1).to(data.device)

    with torch.no_grad():
        next_q_values = data.target_model(next_states).max(dim=1).values
        target_q_values = rewards + data.gamma * next_q_values * (1.0 - dones)

    loss = torch.nn.functional.smooth_l1_loss(current_q_values.to(data.device), target_q_values.to(data.device)).to(data.device)

    data.optimizer.zero_grad()
    loss.backward()
    data.optimizer.step()

In [ ]:
from utils import MoveDirection

from model import get_state_from_board


def run_epoch(data: TrainData) -> None:
    board = get_random_board()
    length: int = 3

    total_score = 0.0
    previous_state = get_state_from_board(board)

    while board.is_running():
        action = select_action(
            data.model,
            previous_state.unsqueeze(0),
            4, data.epsilon_data.current
        )

        board.move(MoveDirection(action))

        reward, length = get_reward(board, length)
        total_score += reward

        new_state: torch.Tensor = torch.zeros_like(previous_state)
        done: bool = False

        if board.get_status() == BoardStatus.RUNNING:
            new_state = get_state_from_board(board)

            if new_state[-1].item() < previous_state[-1].item():
                reward += 5.0
        else:
            done = True
        
        if total_score < -10_000.0:
            break

        data.transition_buffer.append(StateSaver(
            state=previous_state,
            action=action,
            reward=reward,
            next_state=new_state,
            done=done
        ))
        
        optimize_model(data)

        previous_state = new_state
    
    data.epsilon_data.current = max(
        data.epsilon_data.epsilon_end,
        data.epsilon_data.current * data.epsilon_data.epsilon_decay,
    )

In [109]:
from tqdm import tqdm

def epoch_runner(data: TrainData, epoch_number: int) -> None:
    for _ in (bar := tqdm(range(epoch_number))):
        bar.set_description(f"{data.epsilon_data.current:.4f}")
        run_epoch(data)

In [110]:
def save_model(data: TrainData, epoches: int, path: str | None = None) -> None:
    if path is None:
        path = f"../models/result_{epoches}.pt"
    
    torch.save(data.model.state_dict(), path)

In [111]:
def train(epoch_number: int) -> None:
    device = get_device()
    
    model = SnakeSolver(5).to(device)
    target_model = SnakeSolver(5).to(device)
    target_model.load_state_dict(model.state_dict())

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    buffer = Buffer(50_000)

    train_data = TrainData(device, model, target_model, optimizer, buffer)

    try:
        epoch_runner(train_data, epoch_number)
    finally:
        save_model(train_data, epoch_number)

In [112]:
train(6_000)

0.0500: 100%|██████████| 6000/6000 [00:04<00:00, 1235.05it/s]
